In [1]:
# Load dataset

import pandas as pd
import numpy as np
import torch
import pickle
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# Load WNBA dataset
df_master = pd.read_csv('wnba_master_dataset.csv', parse_dates=['GAME_DATE'])
df_master['HOME_AWAY'] = df_master['HOME_AWAY'].map({'HOME': 1, 'AWAY': 0})
df_master['POSITION']  = df_master['POSITION'].map({'G': 0, 'F': 1, 'C': 2})

print(f"Dataset loaded: {len(df_master):,} rows")
print(f"Columns: {len(df_master.columns)}")

Dataset loaded: 14,597 rows
Columns: 92


In [3]:
# Define features

import torch.nn as nn

target_cols = ['PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M']

feature_cols = [
    # 5-game rolling averages
    'PTS_roll5', 'REB_roll5', 'AST_roll5', 'BLK_roll5',
    'STL_roll5', 'FG3M_roll5', 'MIN_roll5', 'TOV_roll5',
    'FGA_roll5', 'FG3A_roll5',

    # 10-game rolling averages
    'PTS_roll10', 'REB_roll10', 'AST_roll10', 'BLK_roll10',
    'STL_roll10', 'FG3M_roll10', 'MIN_roll10', 'TOV_roll10',
    'FGA_roll10', 'FG3A_roll10',

    # Situational
    'HOME_AWAY', 'DAYS_REST', 'DEF_RATING', 'PACE',

    # Opponent stats
    'OPP_PTS_ALLOWED_PG', 'OPP_REB_ALLOWED_PG', 'OPP_AST_ALLOWED_PG',
    'OPP_BLK_PG', 'OPP_STL_PG', 'OPP_3PM_ALLOWED_PG',

    # Usage and position
    'USG_PCT',
    'OPP_PTS_VS_POS', 'OPP_REB_VS_POS', 'OPP_AST_VS_POS',
    'OPP_BLK_VS_POS', 'OPP_STL_VS_POS', 'OPP_3PM_VS_POS',

    # Rolling features
    'USG_PCT_roll5',
    'OPP_PTS_VS_POS_roll5_norm', 'OPP_REB_VS_POS_roll5_norm',
    'OPP_AST_VS_POS_roll5_norm', 'OPP_BLK_VS_POS_roll5_norm',
    'OPP_STL_VS_POS_roll5_norm', 'OPP_3PM_VS_POS_roll5_norm',

    # Team usage context
    'RELATIVE_USG', 'USG_RANK',

    # Volatility
    'PTS_std_roll10', 'REB_std_roll10',
    'PTS_cv_roll10', 'REB_cv_roll10',

    # WNBA specific
    'IS_ROOKIE_SEASON',
    'GAMES_PLAYED',
]

print(f"Total features: {len(feature_cols)}")
print(f"Target outputs: {len(target_cols)}")

Total features: 52
Target outputs: 6


In [5]:
# Add position normalized features and refit scalar

from sklearn.preprocessing import StandardScaler

# Add position-normalized rolling features
pos_roll_cols = [
    'OPP_PTS_VS_POS_roll5', 'OPP_REB_VS_POS_roll5',
    'OPP_AST_VS_POS_roll5', 'OPP_BLK_VS_POS_roll5',
    'OPP_STL_VS_POS_roll5', 'OPP_3PM_VS_POS_roll5'
]

for col in pos_roll_cols:
    norm_col = f'{col}_norm'
    df_master[norm_col] = df_master.groupby('POSITION')[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )

# Load WNBA scaler
with open('wnba_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Verify
df_raw = df_master.dropna(subset=feature_cols).reset_index(drop=True)
test   = df_raw[feature_cols].iloc[0:1].values
norm   = scaler.transform(test)

print(f"Raw PTS_roll5:        {test[0][0]:.4f}")
print(f"Normalized PTS_roll5: {norm[0][0]:.4f}")
print(f"Scaler mean PTS_roll5: {scaler.mean_[0]:.4f}")
print()
print(f"✅ WNBA scaler loaded — {scaler.n_features_in_} features")

Raw PTS_roll5:        17.8000
Normalized PTS_roll5: 1.2546
Scaler mean PTS_roll5: 11.1062

✅ WNBA scaler loaded — 52 features


In [7]:
# Build and load model

class PlayerPropModel(nn.Module):
    def __init__(self, input_dim, target_stats):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.BatchNorm1d(128), nn.Dropout(0.5),
            nn.Linear(128, 64), nn.ReLU(),
            nn.BatchNorm1d(64), nn.Dropout(0.4),
        )
        self.heads = nn.ModuleDict({
            stat: nn.Sequential(
                nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 2)
            ) for stat in target_stats
        })

    def forward(self, x):
        shared = self.trunk(x)
        outputs = {}
        for stat, head in self.heads.items():
            raw       = head(shared)
            mu        = raw[:, 0]
            log_sigma = torch.clamp(raw[:, 1], min=-3, max=3)
            sigma     = torch.exp(log_sigma) + 1e-6
            outputs[stat] = (mu, sigma)
        return outputs

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = PlayerPropModel(input_dim=len(feature_cols), target_stats=target_cols)
model.load_state_dict(torch.load('wnba_best_model.pth', map_location=device))
model.eval()

print(f"WNBA model loaded ✅")
print(f"Device: {device}")
print(f"Input features: {len(feature_cols)}")

WNBA model loaded ✅
Device: cpu
Input features: 52


In [17]:
# Prediction function

def predict_player(player_name, df_master, model, scaler, feature_cols, target_cols):
    """
    Returns predicted (mu, sigma) for each stat.
    Falls back to most recent complete game if latest has NaN features.
    """
    player_df = df_master[df_master['PLAYER_NAME'] == player_name].copy()

    if len(player_df) == 0:
        print(f"❌ Player '{player_name}' not found in dataset")
        return None

    player_df = player_df.sort_values('GAME_DATE')
    latest    = player_df.iloc[-1]

    if player_df[feature_cols].iloc[-1].isna().any():
        complete_rows = player_df.dropna(subset=feature_cols)
        if len(complete_rows) == 0:
            print(f"❌ No complete feature rows for {player_name}")
            return None
        latest = complete_rows.iloc[-1]
        print(f"ℹ️  Using last complete game: {latest['GAME_DATE'].date()}")

    game_count = len(player_df)
    if game_count < 20:
        print(f"⚠️  Low confidence — only {game_count} games in dataset")

    features = latest[feature_cols].values.astype(np.float32).reshape(1, -1)
    features = scaler.transform(features)

    if np.isnan(features).any():
        print(f"❌ NaN values in features for {player_name}")
        return None

    feature_tensor = torch.tensor(features, dtype=torch.float32).to(device)

    with torch.no_grad():
        outputs = model(feature_tensor)

    results = {}
    for stat in target_cols:
        mu    = outputs[stat][0].item()
        sigma = outputs[stat][1].item()
        results[stat] = {'mu': mu, 'sigma': sigma}

    return results, latest['GAME_DATE'], game_count


# Test on A'ja Wilson
results, last_game, count = predict_player(
    "A'ja Wilson", df_master, model, scaler, feature_cols, target_cols
)

print(f"Predictions based on data through: {last_game.date()}")
print(f"Games in dataset: {count}")
print()
print(f"{'Stat':<8} {'Pred μ':>8} {'Pred σ':>8} {'Range (68%)':>20}")
print("-" * 50)
for stat, vals in results.items():
    mu    = vals['mu']
    sigma = vals['sigma']
    print(f"{stat:<8} {mu:>8.1f} {sigma:>8.1f}   {mu-sigma:.1f} – {mu+sigma:.1f}")

Predictions based on data through: 2025-09-11
Games in dataset: 171

Stat       Pred μ   Pred σ          Range (68%)
--------------------------------------------------
PTS          21.5      6.2   15.3 – 27.6
REB          10.3      3.6   6.7 – 13.8
AST           2.6      1.8   0.8 – 4.4
BLK           2.2      1.6   0.6 – 3.8
STL           1.3      1.2   0.2 – 2.5
FG3M          1.1      1.4   -0.3 – 2.5


In [21]:
# Ev Calculator

from scipy.stats import norm as scipy_norm

def calculate_ev(player_name, stat, prop_line, over_juice, under_juice,
                 df_master, model, scaler, feature_cols, target_cols):
    result = predict_player(
        player_name, df_master, model, scaler, feature_cols, target_cols
    )

    if result is None:
        return None

    predictions, last_game, game_count = result

    if stat not in predictions:
        print(f"❌ Stat '{stat}' not available")
        return None

    mu    = predictions[stat]['mu']
    sigma = predictions[stat]['sigma']

    prob_over  = 1 - scipy_norm.cdf(prop_line, mu, sigma)
    prob_under = scipy_norm.cdf(prop_line, mu, sigma)

    def american_to_prob(juice):
        if juice < 0:
            return abs(juice) / (abs(juice) + 100)
        else:
            return 100 / (juice + 100)

    breakeven_over  = american_to_prob(over_juice)
    breakeven_under = american_to_prob(under_juice)

    edge_over  = prob_over  - breakeven_over
    edge_under = prob_under - breakeven_under

    if edge_over > edge_under and edge_over > 0:
        recommendation = 'OVER'
        edge = edge_over
    elif edge_under > edge_over and edge_under > 0:
        recommendation = 'UNDER'
        edge = edge_under
    else:
        recommendation = 'NO BET'
        edge = max(edge_over, edge_under)

    print(f"{'='*50}")
    print(f"  {player_name} — {stat} Prop Analysis")
    print(f"{'='*50}")
    print(f"  Data through:     {last_game.date()}")
    print(f"  Games in dataset: {game_count}")
    print()
    print(f"  Model prediction: μ={mu:.1f}  σ={sigma:.1f}")
    print(f"  Prop line:        {prop_line}")
    print()
    print(f"  Model P(over):    {prob_over:.1%}")
    print(f"  Model P(under):   {prob_under:.1%}")
    print()
    print(f"  Breakeven over:   {breakeven_over:.1%}  (juice: {over_juice})")
    print(f"  Breakeven under:  {breakeven_under:.1%}  (juice: {under_juice})")
    print()
    print(f"  Edge over:        {edge_over:+.1%}")
    print(f"  Edge under:       {edge_under:+.1%}")
    print()

    if recommendation == 'NO BET':
        print(f"  🚫 NO BET — no meaningful edge found")
    else:
        print(f"  ✅ BET {recommendation} {prop_line} {stat}")
        print(f"  Edge: {edge:+.1%}")

    print(f"{'='*50}")

    return {
        'player':         player_name,
        'stat':           stat,
        'line':           prop_line,
        'mu':             mu,
        'sigma':          sigma,
        'prob_over':      prob_over,
        'prob_under':     prob_under,
        'edge_over':      edge_over,
        'edge_under':     edge_under,
        'recommendation': recommendation,
        'edge':           edge,
        'game_count':     game_count,
    }


# Test — A'ja Wilson points prop
result = calculate_ev(
    player_name  = "A'ja Wilson",
    stat         = 'PTS',
    prop_line    = 22.5,
    over_juice   = -110,
    under_juice  = -110,
    df_master    = df_master,
    model        = model,
    scaler       = scaler,
    feature_cols = feature_cols,
    target_cols  = target_cols
)

  A'ja Wilson — PTS Prop Analysis
  Data through:     2025-09-11
  Games in dataset: 171

  Model prediction: μ=21.5  σ=6.2
  Prop line:        22.5

  Model P(over):    43.5%
  Model P(under):   56.5%

  Breakeven over:   52.4%  (juice: -110)
  Breakeven under:  52.4%  (juice: -110)

  Edge over:        -8.9%
  Edge under:       +4.1%

  ✅ BET UNDER 22.5 PTS
  Edge: +4.1%


In [23]:
# Test across multiple WNBA players and stats
test_cases = [
    ("A'ja Wilson",      'PTS',  22.5, -110, -110),
    ("A'ja Wilson",      'REB',  10.5, -110, -110),
    ('Caitlin Clark',    'PTS',  18.5, -110, -110),
    ('Caitlin Clark',    'AST',   7.5, -110, -110),
    ('Paige Bueckers',   'PTS',  19.5, -110, -110),
    ('Sabrina Ionescu',  'PTS',  17.5, -110, -110),
    ('Napheesa Collier', 'PTS',  19.5, -110, -110),
    ('Breanna Stewart',  'REB',   8.5, -110, -110),
]

print("WNBA PROP SCAN RESULTS")
print("=" * 70)

for player, stat, line, over_j, under_j in test_cases:
    result = calculate_ev(
        player_name  = player,
        stat         = stat,
        prop_line    = line,
        over_juice   = over_j,
        under_juice  = under_j,
        df_master    = df_master,
        model        = model,
        scaler       = scaler,
        feature_cols = feature_cols,
        target_cols  = target_cols
    )
    print()

WNBA PROP SCAN RESULTS
  A'ja Wilson — PTS Prop Analysis
  Data through:     2025-09-11
  Games in dataset: 171

  Model prediction: μ=21.5  σ=6.2
  Prop line:        22.5

  Model P(over):    43.5%
  Model P(under):   56.5%

  Breakeven over:   52.4%  (juice: -110)
  Breakeven under:  52.4%  (juice: -110)

  Edge over:        -8.9%
  Edge under:       +4.1%

  ✅ BET UNDER 22.5 PTS
  Edge: +4.1%

  A'ja Wilson — REB Prop Analysis
  Data through:     2025-09-11
  Games in dataset: 171

  Model prediction: μ=10.3  σ=3.6
  Prop line:        10.5

  Model P(over):    47.3%
  Model P(under):   52.7%

  Breakeven over:   52.4%  (juice: -110)
  Breakeven under:  52.4%  (juice: -110)

  Edge over:        -5.1%
  Edge under:       +0.3%

  ✅ BET UNDER 10.5 REB
  Edge: +0.3%

  Caitlin Clark — PTS Prop Analysis
  Data through:     2025-07-15
  Games in dataset: 47

  Model prediction: μ=17.6  σ=6.7
  Prop line:        18.5

  Model P(over):    44.5%
  Model P(under):   55.5%

  Breakeven over:  